In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np 
import seaborn as sns 
%matplotlib inline

In [ ]:
df = pd.read_csv("../data/Zomato Dataset.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df["Weather_conditions"].value_counts(dropna=False)

In [ ]:
df["Road_traffic_density"].value_counts(dropna=False)

In [ ]:
df["Type_of_order"].value_counts(dropna=False)

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(
    data=df,
    x="Time_taken (min)",
    bins=30
)
plt.title("Distribution of food delivery time")
plt.xlabel("Delivery time (minutes)")
plt.ylabel("Number of deliveries")

plt.show()

#as we see number of deliveries generally decreses as delivery time gets longer

In [ ]:
sns.boxplot(
    data=df,
    x="Time_taken (min)",
    y="Road_traffic_density"
)
plt.show()
#heavy traffic is associated with longer delivery time 
#while low traffic has the shortest median deliveruy time.

In [ ]:


def calculate_haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculates the distance (in km) between two points
    on the earth given their latitude and longitude in 
    decimal degrees.
    """     
    R = 6371.0  # Earth radius in kilometers

    # Convert degrees into radians
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    # Difference in coordinates
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Haversine formula (Now properly indented inside the function!)
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


In [ ]:
# Clean coordinates and compute distance
df['Restaurant_latitude'] = df['Restaurant_latitude'].abs()
df['Restaurant_longitude'] = df['Restaurant_longitude'].abs()
df['Delivery_location_latitude'] = df['Delivery_location_latitude'].abs()
df['Delivery_location_longitude'] = df['Delivery_location_longitude'].abs()


In [ ]:
df['distance_km'] = calculate_haversine_distance(
    df['Restaurant_latitude'],
    df['Restaurant_longitude'],
    df['Delivery_location_latitude'],
    df['Delivery_location_longitude']
)


In [ ]:
print(df[['Restaurant_longitude','Restaurant_latitude', 'Delivery_location_latitude', 'distance_km', 'Time_taken (min)']].head())


In [ ]:
df["Delivery_person_Age"]=df["Delivery_person_Age"].fillna(df["Delivery_person_Age"].median())
df["Delivery_person_Ratings"]=df["Delivery_person_Ratings"].fillna(df["Delivery_person_Ratings"].median())
df['multiple_deliveries'] = df['multiple_deliveries'].fillna(df['multiple_deliveries'].median())


In [ ]:
Numerical_cols = [
    "Delivery_person_Age", 
    "Delivery_person_Ratings", 
    "Vehicle_condition", 
    "multiple_deliveries", 
    "distance_km",
    
]

Categorical_cols = [
    "City",
    "Type_of_order",
    "Type_of_vehicle",
    "Weather_conditions",
    "Road_traffic_density",

]

for cols in Categorical_cols:
    df[cols] = df[cols].fillna(df[cols].mode()[0])

In [ ]:
X = df[Numerical_cols+Categorical_cols]
Y = df['Time_taken (min)']


#Train_Test split

In [ ]:
from sklearn.model_selection  import train_test_split
X_train,X_test,Y_train,Y_test= train_test_split(X,Y,random_state=42,test_size=0.2)

In [ ]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
#Create columntransformer and convert it into one hot binary column
preprocessor = ColumnTransformer(
    transformers =[('num', StandardScaler(), Numerical_cols),('cat',OneHotEncoder(sparse_output=False, handle_unknown='ignore'),Categorical_cols)
    ], 
)   


In [ ]:
#Fit and transform the training data and only transform the test data 

X_train_encoded= preprocessor.fit_transform(X_train)
X_test_encoded= preprocessor.transform(X_test)



print(X_train_encoded.shape)
print(X_test_encoded.shape)

In [ ]:
from sklearn.ensemble import RandomForestRegressor


rf_model = RandomForestRegressor(n_estimators=100, max_depth=16,random_state=42,n_jobs=-1)
rf_model.fit(X_train_encoded,Y_train)


In [ ]:
Y_pred_rf = rf_model.predict(X_test_encoded)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(Y_test,Y_pred_rf))
mae = mean_absolute_error(Y_test,Y_pred_rf)
r2 = r2_score(Y_test,Y_pred_rf)

In [ ]:
print(rmse)
print(mae)
print(r2)

In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
import numpy as np

# Predictions on both datasets
y_train_pred = rf_model.predict(X_train_encoded)
y_test_pred = rf_model.predict(X_test_encoded)

# Compare train and test performance
for name, actual, predicted in [
    ("Train", Y_train, y_train_pred),
    ("Test", Y_test, y_test_pred)
]:
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    print(f"\n{name} Performance")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE:  {mae:.2f}")
    print(f"R²:   {r2:.3f}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

rf_10_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=16,
        max_features="sqrt",
        min_samples_leaf=1,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    rf_10_pipeline,
    X_train,
    Y_train,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("Individual CV scores:", cv_scores)
print("Mean CV R2:", round(cv_scores.mean(), 3))
print("Standard deviation:", round(cv_scores.std(), 3))

In [ ]:
rf_10_pipeline.fit(X_train,Y_train)

In [ ]:
from sklearn.metrics import(
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
Y_pred_final = rf_10_pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(Y_test,Y_pred_final))
mae= mean_absolute_error(Y_test,Y_pred_final)
r2 = r2_score(Y_test,Y_pred_final)


print("Final Test RMSE:",round(rmse,2))
print("Final Test MAE:",round(mae,1))
print("Final Test R2:",round(r2,2))

In [ ]:
import os
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(rf_10_pipeline, "../models/food_delivery_model.pkl")

print("Model saved Successfully")


In [ ]:
import joblib
from pathlib import Path

model_path = Path("../models/food_delivery_model.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(rf_10_pipeline, model_path)

print("Final model saved successfully!")